# GPU scaling benchmark: real simulation, estimation & optimization pipelines

Where is the CPU/GPU break-even for the *real* Twin4Build pipelines?  This
notebook sweeps model size N (thermal zones in a chain, consecutive zones
coupled by a wall; compile-time fusion turns the whole chain into ONE
state-space block of `3N-1` states) and times, at every N, on
cpu/fp64, cuda/fp64 and cuda/fp32:

- **Simulation** -- plain forward `Simulator.simulate` (144 steps, no
  gradients, no solver): the baseline cost everything else builds on.
  Metric: **seconds per simulate call** (mean over repeats after a warm-up).
- **Estimation, single shooting** -- `Estimator.estimate` (scipy SLSQP + AD,
  fast single-shooting objective): one `C_air` per zone and one wall `C` per
  wall calibrated against synthetic noisy zone-temperature measurements.
  Each evaluation is a *sequential* 144-step rollout + one reverse sweep.
  Metric: **seconds per objective+gradient evaluation**.
- **Estimation, collocation** -- `Estimator.estimate` (CasADi/IPOPT,
  simultaneous transcription): every timestep-boundary state becomes a
  decision variable and the dynamics are enforced as defects evaluated for
  **all timesteps at once** (batched one-step map) -- far more parallel and
  GPU-friendly than the sequential rollout.
  Metric: **seconds per IPOPT iteration**.
- **Optimization** -- `Optimizer.optimize` (scipy SLSQP + AD, fast composed
  objective): every zone's heater schedule (24 hourly values each) chosen to
  minimize energy under a comfort constraint.  Metric: **seconds per SLSQP
  iteration**.

**Setup**: Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
The full sweep takes roughly 45-60 minutes.  The cuda configurations sweep
up to N=128 zones (383 fused states); cpu rows are capped earlier where the
outcome is no longer in doubt, so the break-even printout is what matters,
not equal-length curves.

**Troubleshooting**: if an import fails with a numpy error after the install,
Runtime > Restart session and Run all again.  To pick up new branch commits,
Runtime > Disconnect and delete runtime first.


In [ ]:
# Clone (or update) the PR branch and install without touching Colab's
# preinstalled scientific stack (replacing numpy mid-session breaks the kernel).
import importlib.metadata as _md
_pins = " ".join(
    f'"{_p}=={_md.version(_p)}"'
    for _p in ("numpy", "scipy", "pandas", "matplotlib")
)
!git clone --depth 1 -b feature/gpu-device-support https://github.com/JBjoernskov/Twin4Build.git 2>/dev/null || git -C Twin4Build pull
!pip install -q ./Twin4Build casadi {_pins}

import sys
sys.path.insert(0, "Twin4Build")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- the sweep will run CPU-only (no break-even measurable).")

CONFIGS = [("cpu", torch.float64)]
if torch.cuda.is_available():
    CONFIGS += [("cuda", torch.float64), ("cuda", torch.float32)]
# CPU rows are capped once the outcome at that size is no longer in doubt
# (a single cpu estimation row at N=128 alone would take ~25 min).
SIM_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]
EST_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]
COL_SIZES = [1, 2, 4, 8, 16, 32, 64]
OPT_SIZES = [1, 2, 4, 8, 16, 32, 64]
SIM_MAX = {"cpu": 128}
EST_MAX = {"cpu": 64}
COL_MAX = {"cpu": 16}
OPT_MAX = {"cpu": 64}


In [ ]:
from twin4build.examples.gpu_benchmark_scaling import (
    breakeven,
    run_estimation_case,
    run_optimization_case,
    run_simulation_case,
    sweep,
)

df_sim = sweep(run_simulation_case, SIM_SIZES, CONFIGS, max_n=SIM_MAX)
df_sim


In [ ]:
df_est = sweep(run_estimation_case, EST_SIZES, CONFIGS, max_n=EST_MAX, maxiter=2)
df_est


In [ ]:
df_col = sweep(
    run_estimation_case, COL_SIZES, CONFIGS, max_n=COL_MAX,
    maxiter=10, transcription="collocation",
)
df_col


In [ ]:
df_opt = sweep(run_optimization_case, OPT_SIZES, CONFIGS, max_n=OPT_MAX, maxiter=5)
df_opt


In [ ]:
import matplotlib.pyplot as plt

panels = [
    (df_sim, "Simulation: s per simulate call"),
    (df_est, "Estimation (single shooting): s per obj+grad eval"),
    (df_col, "Estimation (collocation): s per IPOPT iteration"),
    (df_opt, "Optimization: s per SLSQP iteration"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (df, title) in zip(axes.flat, panels):
    for (device, dtype), grp in df.groupby(["device", "dtype"]):
        grp = grp.sort_values("n_zones")
        ax.plot(
            grp["n_zones"], grp["metric_s"], marker="o",
            label=f"{device}/{dtype}",
        )
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel("zones N  (fused model: 3N-1 states)")
    ax.set_ylabel("wall time [s]")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

for df, name in [
    (df_sim, "simulation"),
    (df_est, "estimation/shooting"),
    (df_col, "estimation/collocation"),
    (df_opt, "optimization"),
]:
    for cfg in [("cuda", "float64"), ("cuda", "float32")]:
        n = breakeven(df, gpu_config=cfg)
        where = f"N >= {n}" if n is not None else "not reached in this sweep"
        print(f"{name:>22} | {cfg[0]}/{cfg[1]:<8} beats cpu/float64 at: {where}")


## How to read the results

- **Simulation** is the pure sequential rollout (144 dependent one-step maps,
  each a `matrix_exp` + mat-vec on the fused `(3N-1)`-state block).  Its
  break-even is the cleanest measure of when raw model size alone justifies
  the GPU.
- **Estimation (single shooting)** adds one reverse-mode sweep per
  evaluation on top of the same sequential rollout -- expect its curves to
  track the simulation panel with a constant factor.
- **Estimation (collocation)** evaluates the continuity defects for all
  timesteps *in one batched call* (`vmap` over the one-step map), so its
  per-iteration GPU work is a few large kernels instead of hundreds of tiny
  ones -- watch `gpu_util_pct`: it should be markedly higher than in the
  shooting panels, and the break-even earlier.  Note the wall time includes
  the one-time transcription setup, so few-iteration runs overstate
  s/iteration; IPOPT itself always runs on the CPU.
- **Small N**: per-op work is microseconds; the GPU pays a kernel-launch
  latency (~5-10 us) on every op and loses.  This region belongs to the CPU.
- **Growing N**: the `O(n^3)` `matrix_exp` starts to dominate and the GPU's
  arithmetic advantage compounds -- the cuda curves flatten while the cpu
  curve rises.  The printed break-even is where they cross.
- **fp32 vs fp64**: on consumer GPUs (T4: fp64 at 1/32 rate) the fp32 curve
  should cross substantially earlier -- but only once the GPU is actually
  compute-bound.  While `gpu_util_pct` is low, launch overhead dominates and
  fp32 buys almost nothing.
- **`gpu_util_pct`** is the fraction of wall-clock time the GPU was actually
  executing a kernel (NVML `utilization.gpu`, sampled at 10 Hz during the
  run; NaN for cpu rows).  At small N expect single-digit percentages --
  the run is dominated by Python overhead and kernel-launch latency, the GPU
  mostly idles.  The break-even N is roughly where this number gets large:
  once the GPU is busy most of the wall time, adding states is nearly free
  for the cuda curves while the cpu curve keeps rising.
- Not covered here: *batched* workloads (multi-start, scenarios, portfolios
  via `n_c`), where the GPU wins at much smaller per-model sizes -- see
  `gpu_benchmark_estimation.ipynb` / `gpu_benchmark_optimizer.ipynb` Part B.
